<a href="https://colab.research.google.com/github/DiyaMalik07/SC_Project/blob/main/spaSC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install -q pymoo

import numpy as np
import pandas as pd
import time
import urllib.request

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score

from pymoo.core.problem import Problem
from pymoo.core.sampling import Sampling
from pymoo.algorithms.moo.spea2 import SPEA2
from pymoo.operators.crossover.hux import HalfUniformCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.optimize import minimize

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [13]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/internet_ads/ad.data"
urllib.request.urlretrieve(url, "ad.data")

df = pd.read_csv("ad.data", header=None, low_memory=False)
print(f"Dataset shape (rows, columns): {df.shape}")

Dataset shape (rows, columns): (3279, 1559)


In [14]:
print("Preprocessing data")

df = df.replace(r'^\s*\?\s*$', np.nan, regex=True)

label_col = df.columns[-1]
y = df[label_col].astype(str).str.strip().apply(
    lambda v: 1 if v.lower().startswith('ad') else 0
).values

X = df.drop(columns=[label_col])
X = X.apply(pd.to_numeric, errors='coerce')
X = X.fillna(X.mean())
X = X.values

n_features = X.shape[1]

print(f"Total features: {n_features}")
print(f"Total instances: {X.shape[0]}")
print(f"Class balance -> ad: {sum(y==1)}  |  nonad: {sum(y==0)}")

Preprocessing data
Total features: 1558
Total instances: 3279
Class balance -> ad: 459  |  nonad: 2820


In [15]:
print("\nSTEP 3: Splitting data (Train 60% / Validation 20% / Test 20%)")

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_temp
)

print(f"Train samples: {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")


STEP 3: Splitting data (Train 60% / Validation 20% / Test 20%)
Train samples: 1967
Validation samples: 656
Test samples: 656


In [16]:
print("Training Random Forest on ALL features")

start_time = time.time()
baseline_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
baseline_model.fit(X_train, y_train)
baseline_train_time = time.time() - start_time

baseline_predictions = baseline_model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_f1 = f1_score(y_test, baseline_predictions)

print(f"Baseline Accuracy : {baseline_accuracy:.4f}")
print(f"Baseline F1-score : {baseline_f1:.4f}")
print(f"Features used     : {n_features}")
print(f"Training time     : {baseline_train_time:.2f} seconds")

Training Random Forest on ALL features
Baseline Accuracy : 0.9665
Baseline F1-score : 0.8690
Features used     : 1558
Training time     : 1.53 seconds


In [17]:
print("Setting up SPEA2 feature selection")
class DiverseBinarySampling(Sampling):
    def _do(self, problem, n_samples, **kwargs):
        population = np.zeros((n_samples, problem.n_var), dtype=bool)
        for i in range(n_samples):
            selection_prob = np.random.uniform(0.02, 0.80)
            population[i] = np.random.rand(problem.n_var) < selection_prob
            if population[i].sum() == 0:
                population[i, np.random.randint(0, problem.n_var)] = True
        return population


class FeatureSelectionProblem(Problem):
    def __init__(self, X_train, y_train, X_val, y_val):
        super().__init__(n_var=n_features, n_obj=2, n_constr=0, xl=0, xu=1, vtype=bool)
        self.X_train, self.y_train = X_train, y_train
        self.X_val, self.y_val = X_val, y_val

    def _evaluate(self, population, out, *args, **kwargs):
        f1_loss_list = []
        feature_fraction_list = []

        for individual in population:
            mask = individual.astype(bool)
            if mask.sum() == 0:
                mask[np.random.randint(0, n_features)] = True

            model = RandomForestClassifier(n_estimators=30, random_state=RANDOM_SEED, n_jobs=-1)
            model.fit(self.X_train[:, mask], self.y_train)
            predictions = model.predict(self.X_val[:, mask])

            f1 = f1_score(self.y_val, predictions)
            f1_loss_list.append(1 - f1)
            feature_fraction_list.append(mask.sum() / n_features)

        out["F"] = np.column_stack([f1_loss_list, feature_fraction_list])


problem = FeatureSelectionProblem(X_train, y_train, X_val, y_val)

Setting up SPEA2 feature selection


In [18]:
print("\nRunning SPEA2 algorithm")

POPULATION_SIZE = 40
GENERATIONS = 30

spea2_algorithm = SPEA2(
    pop_size=POPULATION_SIZE,
    sampling=DiverseBinarySampling(),
    crossover=HalfUniformCrossover(),
    mutation=BitflipMutation(prob=5.0 / n_features),
    eliminate_duplicates=True
)

start_time = time.time()
result = minimize(
    problem,
    spea2_algorithm,
    termination=('n_gen', GENERATIONS),
    seed=RANDOM_SEED,
    verbose=True
)
spea2_run_time = time.time() - start_time

print(f"\nSPEA2 finished in {spea2_run_time:.2f} seconds")


Running SPEA2 algorithm
n_gen  |  n_eval  | n_nds  |      eps      |   indicator  
     1 |       40 |     11 |             - |             -
     2 |       80 |     14 |  0.0230124628 |             f
     3 |      120 |     14 |  0.0276494357 |             f
     4 |      160 |     14 |  0.0102012900 |             f
     5 |      200 |     17 |  0.3664516129 |         nadir
     6 |      240 |     17 |  0.0224383402 |             f
     7 |      280 |     18 |  0.0103267230 |             f
     8 |      320 |     17 |  0.0098334165 |             f
     9 |      360 |     19 |  0.0096933956 |             f
    10 |      400 |     24 |  0.0058600464 |             f
    11 |      440 |     18 |  0.0027967470 |             f
    12 |      480 |     18 |  0.0105170792 |             f
    13 |      520 |     12 |  0.0265286555 |             f
    14 |      560 |     15 |  0.0449412871 |         nadir
    15 |      600 |     13 |  0.0192264924 |             f
    16 |      640 |     13 |  0

In [19]:
print("\nSelecting best feature subset from Pareto front")

pareto_objectives = result.F
pareto_solutions = result.X

validation_f1_scores = 1 - pareto_objectives[:, 0]
selected_feature_counts = (pareto_objectives[:, 1] * n_features).astype(int)

pareto_table = pd.DataFrame({
    "Validation_F1": validation_f1_scores,
    "Num_Features": selected_feature_counts
})

baseline_val_predictions = baseline_model.predict(X_val)
baseline_val_f1 = f1_score(y_val, baseline_val_predictions)
min_acceptable_f1 = 0.95 * baseline_val_f1

good_solutions = pareto_table[pareto_table["Validation_F1"] >= min_acceptable_f1]

if len(good_solutions) > 0:
    chosen_index = good_solutions.sort_values("Num_Features").index[0]
else:
    chosen_index = pareto_table.sort_values("Validation_F1", ascending=False).index[0]

selected_feature_mask = pareto_solutions[chosen_index].astype(bool)
print(f"SPEA2 selected {selected_feature_mask.sum()} features (out of {n_features})")


Selecting best feature subset from Pareto front
SPEA2 selected 38 features (out of 1558)


In [20]:
print("\nTraining Random Forest again, using ONLY SPEA2-selected features")

start_time = time.time()
final_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
final_model.fit(X_train[:, selected_feature_mask], y_train)
final_train_time = time.time() - start_time

final_predictions = final_model.predict(X_test[:, selected_feature_mask])
final_accuracy = accuracy_score(y_test, final_predictions)
final_f1 = f1_score(y_test, final_predictions)

print(f"Final Accuracy (selected features) : {final_accuracy:.4f}")
print(f"Final F1-score (selected features) : {final_f1:.4f}")


Training Random Forest again, using ONLY SPEA2-selected features
Final Accuracy (selected features) : 0.9588
Final F1-score (selected features) : 0.8364


In [21]:
print("FINAL RESULTS: RANDOM FOREST -- BEFORE vs AFTER SPEA2")
print("\n")
print(f"{'Metric':<28}{'All Features':<18}{'SPEA2-Selected':<18}")
print("-" * 60)
print(f"{'Accuracy':<28}{baseline_accuracy:<18.4f}{final_accuracy:<18.4f}")
print(f"{'F1-score':<28}{baseline_f1:<18.4f}{final_f1:<18.4f}")
print(f"{'Number of Features':<28}{n_features:<18}{int(selected_feature_mask.sum()):<18}")
reduction_pct = (1 - selected_feature_mask.sum() / n_features) * 100
print(f"{'Feature Reduction %':<28}{'-':<18}{reduction_pct:<18.1f}")
print(f"{'Training Time (s)':<28}{baseline_train_time:<18.2f}{final_train_time:<18.2f}")

FINAL RESULTS: RANDOM FOREST -- BEFORE vs AFTER SPEA2


Metric                      All Features      SPEA2-Selected    
------------------------------------------------------------
Accuracy                    0.9665            0.9588            
F1-score                    0.8690            0.8364            
Number of Features          1558              38                
Feature Reduction %         -                 97.6              
Training Time (s)           1.53              0.48              
